# Day 35 Tutorial：GNN 数据就绪评审

## Goal

根据图对象、边语义、标签、划分、样本量和基线作出可复查决策。

## Setup

使用人工数据集概况，不训练 GNN；门槛缺失时允许得到 No-Go。

In [1]:
import pandas as pd

dataset_profile = {
    'graph_object_defined': True,
    'node_features_traceable': True,
    'edge_semantics_defined': True,
    'label_protocol_defined': False,
    'leakage_safe_split_defined': False,
    'sample_size_reviewed': True,
    'simple_baseline_available': False,
}
dataset_profile

{'graph_object_defined': True,
 'node_features_traceable': True,
 'edge_semantics_defined': True,
 'label_protocol_defined': False,
 'leakage_safe_split_defined': False,
 'sample_size_reviewed': True,
 'simple_baseline_available': False}

## Steps

### 1. 逐项生成门槛表

In [2]:
next_steps = {
    'label_protocol_defined': 'freeze target level and metric',
    'leakage_safe_split_defined': 'group related graphs before splitting',
    'simple_baseline_available': 'run Dummy and non-GNN baselines',
}
gate_table = pd.DataFrame([
    {
        'gate': name,
        'passed': passed,
        'next_step': 'none' if passed else next_steps.get(name, 'review evidence'),
    }
    for name, passed in dataset_profile.items()
])
gate_table

,gate,passed,next_step
0,graph_object_defined,True,none
1,node_features_traceable,True,none
2,edge_semantics_defined,True,none
3,label_protocol_defined,False,freeze target level and metric
4,leakage_safe_split_defined,False,group related graphs before splitting
5,sample_size_reviewed,True,none
6,simple_baseline_available,False,run Dummy and non-GNN baselines


### 2. 让决策来自证据

In [3]:
blocking = gate_table.loc[~gate_table['passed'], 'gate'].tolist()
decision = 'Go' if not blocking else 'No-Go'
print('decision:', decision)
print('blocking:', blocking)

decision: No-Go
blocking: ['label_protocol_defined', 'leakage_safe_split_defined', 'simple_baseline_available']


## Checks

确认决策、缺口和下一步一一对应。

In [4]:
assert decision == 'No-Go'
assert set(blocking) == {
    'label_protocol_defined',
    'leakage_safe_split_defined',
    'simple_baseline_available',
}
assert gate_table.loc[~gate_table['passed'], 'next_step'].ne('none').all()
print('Checks passed: decision follows the readiness gates.')

Checks passed: decision follows the readiness gates.


## Next Steps

补齐标签协议、分组划分和简单基线后重新评审；不要删除门槛来强行得到 Go。